In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os, glob
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.pyplot import cm

def calc_parc_o3(var):
    
        m_air = 28.964/(6.022E23)
        g = 980.6
    
        if 'plev' in var.dims: 
            plev=var.plev
            level='plev'
        if 'lev' in var.dims: 
            plev=var.lev
            level='lev'
        if 'level' in var.dims: 
            plev=var.level
            level='level'
        
        
        time=var.time
        delta_p = np.zeros((len(time), len(plev)))
        
        if plev[0]<plev[len(plev)-1] and plev[len(plev)-1] <= 1000: 
            factor=100
            factor_2=1 # conversion Pa->hPa
        if plev[0]>plev[len(plev)-1] and plev[0] <=1000: 
            factor=100
            factor_2=1
        if plev[0]<plev[len(plev)-1] and plev[len(plev)-1] >1000: 
            factor=1
            factor_2=100
        if plev[0]>plev[len(plev)-1] and plev[0] >1000: 
            factor=1
            factor_2=100
        
        if plev[0]<plev[len(plev)-1]: # for pressure levels in hPa
            
            
            for levelx in range(1,len(plev)): delta_p[:,levelx].fill(plev[levelx] - plev[levelx-1])

            weights_p = xr.DataArray(delta_p*factor, dims=['time',level], coords=[time,plev]) # difference between pressure levels in Pa
 
            O3 = var * weights_p * 10/ (g * m_air)
        
            if level=='plev': O3=O3.sel(plev=slice(30*factor_2,70*factor_2)) 
            if level=='lev': O3=O3.sel(lev=slice(30*factor_2,70*factor_2))
            if level=='level': O3=O3.sel(level=slice(30*factor_2,70*factor_2))

            O3 = O3.sum(dim=level)
            O3 = O3/2.687E16  #calculate DU
            
        if plev[0]>plev[len(plev)-1]: # for pressure levels in hPa
        
            
            for levelx in range(0,len(plev)-1): delta_p[:,levelx].fill(plev[levelx] - plev[levelx+1])

            weights_p = xr.DataArray(delta_p*factor, dims=['time',level], coords=[time,plev]) # difference between pressure levels in Pa

            O3 = var * weights_p * 10/ (g * m_air)
            
            if level =='plev': O3=O3.sel(plev=slice(70*factor_2,30*factor_2)) 
            if level=='lev': O3=O3.sel(lev=slice(70*factor_2,30*factor_2)) 
            if level=='level': O3=O3.sel(level=slice(70*factor_2,30*factor_2)) 
                
            O3 = O3.sum(dim=level)
            O3 = O3/2.687E16  #calculate DU
    
        return O3.where(O3 != 0)
#========================================================
# Block 1 — 读取 200 年 O3 数据，并计算 60–90°N, 30–70 hPa 部分臭氧柱 (DU)
#   - 使用你已有的 calc_parc_o3(var) 函数
#   - 结果 var: time 序列 (日), 极帽加权平均后的 partial column (DU)
#========================================================

# 1) 读取 200 年 O3 (0101–0300, 每年一个文件)
o3_pattern = "/mnt/backup_ETH/EXTR_2000/CO2x1SmidEmin_yBWCN/atm/hist/h1/o3/CO2x1SmidEmin_yBWCN.cam.h1.????.O3.isobar.nc"
o3_files = sorted(glob.glob(o3_pattern))
print(f"Found {len(o3_files)} O3 files:", o3_files[0], "...", o3_files[-1])

# 多文件拼接：time 维度拼一起
ds_o3 = xr.open_mfdataset(
    o3_files,
    combine="by_coords",
    parallel=False
)

# 如果需要确认维度名，可以打印一次
# print(ds_o3)

#--------------------------------------------------------
# 2) 计算 30–70 hPa 部分臭氧柱 (DU) —— 使用你已有的 calc_parc_o3
#    注意：这里把原始四维 O3 传进去 (time, lev/plev, lat, lon)
#--------------------------------------------------------
O3_4d = ds_o3["O3"]        # (time, plev, lat, lon) — isobaric levels
O3_parc = calc_parc_o3(O3_4d)   # (time, lat, lon), 30–70 hPa 部分柱，单位 DU

#--------------------------------------------------------
# 3) 纬向平均 & 极帽平均 (60–90°N, 余弦加权)
#--------------------------------------------------------

# 先经向平均
O3_zm = O3_parc.mean(dim="lon")        # (time, lat)

# 选 60–90°N
var = O3_zm.sel(lat=slice(60, 90))     # (time, lat)

# 余弦权重
weights = np.cos(np.deg2rad(var.lat))
var = var.weighted(weights).mean(dim="lat")   # (time,) 极帽平均



#--------------------------------------------------------
# 5) 基本信息：总年数
#--------------------------------------------------------
years_200 = int(var.time.dt.year.max() - var.time.dt.year.min() + 1)
print(f"Total years in 200-yr run: {years_200}")
# Block1 最后：
var = var.load()   # 或者 var = var.compute()
print("var computed & loaded into memory, shape:", var.shape)



In [ ]:
def find_ozone_extremes(nc_fid, var, lev, years, extreme_years, model):
    """
    在多年份 O3 数据中，基于 3–4 月的日值，寻找：
      - 每年的春季最大值 / 最小值
      - 在所有年份中，选出 extreme_years 个最高 O3 年和最低 O3 年
    """

    partial_column = True  # True: 用 30–70 hPa partial column；False: 用 70 hPa mixing ratio

    O3 = nc_fid[var]
    plev = nc_fid[lev]
    lev_dim = lev

    time = nc_fid["time"]

    # 插值到 2.5° 纬度
    O3 = O3.interp(lat=np.linspace(-90, 90, 73))

    if partial_column:
        # 计算 30–70 hPa 极帽 partial column

        delta_p = np.zeros((len(O3.time), len(plev)))

        m_air = 28.964 / (6.022E23)
        g = 980.6

        # 情况 1：气压单位是 hPa，且自下而上递增
        if plev[len(plev)-1] <= 1000 and model != "MERRA":
            for level_i in range(1, len(plev)):
                delta_p[:, level_i].fill(plev[level_i] - plev[level_i-1])

            O3 = O3.sel(lat=slice(60, 90))
            weights = np.cos(np.deg2rad(O3.lat))
            O3 = O3.weighted(weights).mean(dim="lat")

            weights_p = xr.DataArray(
                delta_p * 100,
                dims=["time", lev_dim],
                coords={"time": time, lev_dim: plev}
            )

            O3 = O3 * weights_p * 10 / (g * m_air)

            O3 = O3.sel({lev_dim: slice(30, 70)})
            O3 = O3.sum(dim=lev_dim)

            O3 = O3 / 2.687E16
            plev = plev.sel({lev_dim: slice(30, 70)})

        # 情况 2：气压单位是 Pa，且自下而上递增
        if plev[len(plev)-1] > 1000 and model != "MERRA":
            for level_i in range(1, len(plev)):
                delta_p[:, level_i].fill(plev[level_i] - plev[level_i-1])

            O3 = O3.sel(lat=slice(60, 90))
            weights = np.cos(np.deg2rad(O3.lat))
            O3 = O3.weighted(weights).mean(dim="lat")

            weights_p = xr.DataArray(
                delta_p,
                dims=["time", lev_dim],
                coords={"time": time, lev_dim: plev}
            )

            O3 = O3 * weights_p * 10 / (g * m_air)

            O3 = O3.sel({lev_dim: slice(3000, 7000)})
            O3 = O3.sum(dim=lev_dim)

            O3 = O3 / 2.687E16

            plev = plev.sel({lev_dim: slice(3000, 7000)})
            plev = plev / 100  # 转 hPa

        # 情况 3：MERRA
        if model == "MERRA":
            for level_i in range(0, len(plev)-1):
                delta_p[:, level_i].fill(plev[level_i] - plev[level_i+1])

            O3 = O3.sel(lat=slice(60, 90)) * 28.970 / 47.9982
            weights = np.cos(np.deg2rad(O3.lat))
            O3 = O3.weighted(weights).mean(dim="lat")

            weights_p = xr.DataArray(
                delta_p * 100,
                dims=["time", lev_dim],
                coords={"time": time, lev_dim: plev}
            )

            O3 = O3 * weights_p * 10 / (g * m_air)

            O3 = O3.sel({lev_dim: slice(70, 30)})
            O3 = O3.sum(dim=lev_dim)

            O3 = O3 / 2.687E16
            plev = plev.sel({lev_dim: slice(70, 30)})

    if not partial_column:
        if plev[len(plev)-1] > 1000 and model != "MERRA":
            O3 = O3.sel(lat=slice(60, 90)).sel({lev_dim: 7000})
        if plev[len(plev)-1] <= 1000 and model != "MERRA":
            O3 = O3.sel(lat=slice(60, 90)).sel({lev_dim: 70})
        if model == "MERRA":
            O3 = O3.sel(lat=slice(60, 90)).sel({lev_dim: 70})

        weights = np.cos(np.deg2rad(O3.lat))
        O3 = O3.weighted(weights).mean(dim="lat") * 1e6  # ppmv

    # -------- 选 3–4 月 --------
    O3_clim = O3.groupby("time.month").mean()  # 虽然后面没用，但留着不动

    O3 = O3.sel(time=nc_fid.time.dt.month.isin([3, 4]))
    time_ma = time.sel(time=nc_fid.time.dt.month.isin([3, 4]))

    # -------- 按年分组 --------
    grouped_O3 = O3.groupby("time.year")
    grouped_time = time_ma.groupby("time.year")

    years = len(grouped_O3)

    O3_highest_values = np.zeros(years)
    O3_lowest_values = np.zeros(years)
    O3_highest_indices = np.zeros(years, dtype=int)
    O3_lowest_indices = np.zeros(years, dtype=int)
    O3_lowest_dates = []
    O3_highest_dates = []
    O3_years = np.zeros((years,))

    # ✅ 在这里把 “所有非 time 维度” 先平均掉，保证 arr 是 1D (N_days,)
    for i, ((year, da_O3), (_, da_time)) in enumerate(zip(grouped_O3, grouped_time)):
        if "time" not in da_O3.dims:
            raise ValueError(f"O3 group for year {year} has no 'time' dimension, dims={da_O3.dims}")

        other_dims = [d for d in da_O3.dims if d != "time"]
        if other_dims:
            da_O3_1d = da_O3.mean(dim=other_dims)
        else:
            da_O3_1d = da_O3

        arr = da_O3_1d.values  # shape: (N_spring_days,)

        # 防止全 NaN 的极端情况（很不可能，但以防万一）
        if np.all(np.isnan(arr)):
            h_idx = 0
            l_idx = 0
        else:
            h_idx = int(np.nanargmax(arr))
            l_idx = int(np.nanargmin(arr))

        O3_highest_values[i] = arr[h_idx]
        O3_lowest_values[i] = arr[l_idx]
        O3_highest_indices[i] = h_idx
        O3_lowest_indices[i] = l_idx

        t_arr = da_time.values
        O3_highest_dates.append(np.array(t_arr[h_idx]))
        O3_lowest_dates.append(np.array(t_arr[l_idx]))

        O3_years[i] = year

    # -------- 在所有年份中选出 extreme_years 个最高/最低春季 O3 年 --------
    O3_highest = O3_highest_values.argsort()[-extreme_years:][::-1]
    O3_lowest = O3_lowest_values.argsort()[0:extreme_years]

    O3_lowest_index = np.zeros((extreme_years), dtype=int)
    O3_highest_index = np.zeros((extreme_years), dtype=int)
    O3_lowest_date_sel = []
    O3_highest_date_sel = []

    for i in range(extreme_years):
        O3_lowest_index[i] = O3_lowest_indices[O3_lowest[i]]
        O3_highest_index[i] = O3_highest_indices[O3_highest[i]]

        O3_lowest_date_sel.append(O3_lowest_dates[O3_lowest[i]])
        O3_highest_date_sel.append(O3_highest_dates[O3_highest[i]])

    print(
        "Mean minimum ozone day: "
        + str(np.mean(O3_lowest_index))
        + " ± "
        + str(np.std(O3_lowest_index))
    )
    print(
        "Mean maximum ozone day: "
        + str(np.mean(O3_highest_index))
        + " ± "
        + str(np.std(O3_highest_index))
    )

    O3_intersect = len(np.intersect1d(O3_highest, O3_lowest))

    return (
        O3_highest,
        O3_lowest,
        np.reshape(O3_lowest_date_sel, (extreme_years,)),
        np.reshape(O3_highest_date_sel, (extreme_years,)),
        np.reshape(O3_lowest_index, (extreme_years,)),
        np.reshape(O3_highest_index, (extreme_years,)),
        O3_intersect,
        O3_years,
    )


#========================================================
# Block 2
# 1) 用 200 年 O3 数据找极端低 O3 年（基于 3–4 月日值）
# 2) 对这些年份构造 “前一年的 10–12 月 + 该年的 1–9 月” 序列
# 3) 与非极端年 climatology 做对比，并画出 O3 演变图
#
# 依赖：
#   - var   (Block 1 算出的 60–90°N, 30–70 hPa partial column, DU)
#   - ds_o3 (200 年 O3 原始 isobar 数据)
#   - find_ozone_extremes (你原来的函数)
#========================================================

fig, ax = plt.subplots(
    1, 1,
    figsize=(13, 10),
    constrained_layout=True
)

extreme_years = 10      # 依然选 10 个最低 O3 年
model = "WACCM"
lev_name = "lev"    # ← 这里这样写
O3_varname = "O3"

years_200 = len(np.unique(ds_o3.time.dt.year.values))

O3_highest, O3_lowest, O3_lowest_date, O3_highest_date, \
O3_lowest_index, O3_highest_index, O3_intersect, O3_years = \
    find_ozone_extremes(
        ds_o3,
        O3_varname,
        lev_name,      # 这里传 'lev'
        years_200,
        extreme_years,
        model
    )


print("O3_years array:", O3_years)
print("Indices of lowest-O3 years:", O3_lowest)

# 映射成真正的年份标签（如 0101, 0102, …）
O3_lowest_years = O3_years[O3_lowest.astype(int)]
print("Lowest-O3 years (labels):", O3_lowest_years)

#--------------------------------------------------------
# 2) 划分“非极端年”（neutral years）
#--------------------------------------------------------
O3_neutral_years = []

for y in O3_years:
    if not np.isin(y, O3_lowest_years):
        O3_neutral_years.append(y)

O3_neutral_years = np.array(O3_neutral_years)

#--------------------------------------------------------
# 3) 按年选出时间序列，并计算 day-of-year climatology
#    var: (time,) 200 年 × 365 天 的 partial column (DU)
#--------------------------------------------------------

# 低 O3 年
var_lowest = var.sel(time=var.time.dt.year.isin(O3_lowest_years))
var_lowest_mean = np.array(
    var_lowest.groupby("time.dayofyear").mean("time")
)
var_lowest_std = np.array(
    var_lowest.groupby("time.dayofyear").std("time")
)

# 低 O3 年的前一年
var_lowest_before = var.sel(time=var.time.dt.year.isin(O3_lowest_years - 1))
var_lowest_mean_before = np.array(
    var_lowest_before.groupby("time.dayofyear").mean("time")
)
var_lowest_std_before = np.array(
    var_lowest_before.groupby("time.dayofyear").std("time")
)

# 非极端年
var_neutral = var.sel(time=var.time.dt.year.isin(O3_neutral_years))
var_neutral_mean = np.array(
    var_neutral.groupby("time.dayofyear").mean("time")
)
var_neutral_std = np.array(
    var_neutral.groupby("time.dayofyear").std("time")
)

# 非极端年的前一年
var_neutral_before = var.sel(time=var.time.dt.year.isin(O3_neutral_years - 1))
var_neutral_mean_before = np.array(
    var_neutral_before.groupby("time.dayofyear").mean("time")
)
var_neutral_std_before = np.array(
    var_neutral_before.groupby("time.dayofyear").std("time")
)

#--------------------------------------------------------
# 4) reshape 成 (extreme_years, 365)，得到逐年曲线
#    假设 noleap → 每年 365 天
#--------------------------------------------------------
var_lowest = np.reshape(np.array(var_lowest), (extreme_years, 365))
var_lowest_before = np.reshape(np.array(var_lowest_before), (extreme_years, 365))

#--------------------------------------------------------
# 5) 画图：前一年 10–12 月 (x=0–90) + 该年 1–9 月 (x=91–364)
#    颜色方案与原代码类似
#--------------------------------------------------------

color = cm.twilight(np.linspace(0, 1, extreme_years))

for j in range(extreme_years):
    # 低 O3 年 1–9 月 (映射到 day 91–364)
    ax.plot(
        range(91, 365),
        var_lowest[j, 0:365-91],
        color=color[j],
        alpha=0.8,
        linewidth=2,
        label='low O3 years' if j == 0 else None
    )

    # 低 O3 年前一年的 10–12 月 (映射到 day 0–90)
    ax.plot(
        range(0, 91),
        var_lowest_before[j, 365-92:365-1],
        color=color[j],
        alpha=0.8,
        linewidth=2
    )

# 非极端年 climatology ± std （灰色带）
ax.errorbar(
    range(91, 365),
    var_neutral_mean[0:365-91],
    var_neutral_std[0:365-91],
    color='grey',
    alpha=0.5,
    elinewidth=1.5,
    linewidth=3,
    label='all other years'
)

ax.errorbar(
    range(0, 91),
    var_neutral_mean_before[365-92:365-1],
    var_neutral_std_before[365-92:365-1],
    color='grey',
    alpha=0.5,
    elinewidth=1.5,
    linewidth=3
)

#--------------------------------------------------------
# 6) 轴 & 样式 & 保存
#--------------------------------------------------------
ax.set_xlim(0, 366)

ax.set_xticks([0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334])
ax.set_xticklabels(
    ['Oct', 'Nov', 'Dec', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'June', 'July', 'Aug', 'Sep'],
    fontsize=15
)

ax.set_ylabel('Partial ozone column, 30–70 hPa (DU)', fontsize=18)
ax.set_yticklabels(ax.get_yticks(), size=18)
ax.legend(fontsize=14)

out_dir = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3"
os.makedirs(out_dir, exist_ok=True)

plt.savefig(os.path.join(out_dir, "O3_evolution_min_10extr_200yrs.png"), dpi=300)
plt.savefig(os.path.join(out_dir, "O3_evolution_min_10extr_200yrs.pdf"))

plt.show()


In [ ]:
# ========================================================
# Block 3 — 数据读取与处理：
#   1. 读取 merged WACCM 文件，提取 0008/0013/0014/0019 年
#   2. 用 200 年 EXTR O3 部分柱 (var) 算：
#       - 全部年份月气候态 (clim_all)
#       - 低臭氧年份（最低 25%）的月“合成”气候态 (clim_low)
# ========================================================

import os
import numpy as np
import xarray as xr

# ---------- 3.1 读取 merged WACCM 文件，并计算 30–70 hPa 极帽部分柱 ----------

merged_path = "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc"
ds_merged = xr.open_dataset(merged_path)

# 提取 O3，计算 30–70 hPa 部分柱 (DU)
O3_merged_4d = ds_merged["O3"]       # (time, lev, lat, lon) or similar
O3_merged_parc = calc_parc_o3(O3_merged_4d)  # (time, lat, lon), 30–70 hPa partial column in DU

# 先经向平均
O3_merged_zm = O3_merged_parc.mean(dim="lon")  # (time, lat)

# 极帽 60–90°N 余弦加权平均
var_merged = O3_merged_zm.sel(lat=slice(60, 90))
weights_merged = np.cos(np.deg2rad(var_merged.lat))
var_merged = var_merged.weighted(weights_merged).mean(dim="lat")  # (time,)

# 检查一下年份范围（方便你对照 0008/0013/0014/0019 是什么样的 year）
print("Merged run unique years:", np.unique(var_merged.time.dt.year.values))

# 你给出的目标年份：0008, 0013, 0014, 0019
# 在 xarray 里通常是整数 8, 13, 14, 19，如果实际是 2008/2013 等，请改成对应值
years_special = np.array([8, 13, 14, 19], dtype=int)

# 为防止选不到任何数据，可以交叉检查一下：
available_years = np.unique(var_merged.time.dt.year.values)
print("Available years in merged:", available_years)

missing = years_special[~np.isin(years_special, available_years)]
if len(missing) > 0:
    print("⚠️ 这些目标年份在 merged 文件中不存在，请根据 available_years 调整：", missing)

# 逐年计算月平均 (12 个点)
months = np.arange(1, 13, dtype=int)
monthly_pc_merged_list = []

for y in years_special:
    da_y = var_merged.sel(time=var_merged.time.dt.year == y)
    # 对该年的 12 个月分别取时间平均
    da_y_month = da_y.groupby("time.month").mean("time")
    # 为了确保有 12 个月，reindex 一下（如果缺某月会是 NaN）
    da_y_month = da_y_month.reindex(month=months)
    monthly_pc_merged_list.append(da_y_month.values)

# 拼成 DataArray: (year, month)
monthly_pc_merged = xr.DataArray(
    np.stack(monthly_pc_merged_list, axis=0),
    dims=("year", "month"),
    coords={"year": years_special, "month": months},
    name="O3_partial_column_30_70hPa_60_90N"
)

print("monthly_pc_merged shape:", monthly_pc_merged.shape)
# 现在 monthly_pc_merged[y_idx, m_idx] 对应某年某月的部分柱 (DU)


# ---------- 3.2 基于 200 年 EXTR run 的 var 计算月气候态 + 低臭氧年份合成 ----------

# 这里假设 Block1 已经算出 var: 200 年 × 365 天 的 60–90N 30–70hPa 部分柱 (DU)
# 如果你是在新 notebook 里使用，记得先运行 Block1 得到 var

# 200 年总年份数（通常是 200）
years_extr_all = np.unique(var.time.dt.year.values)
n_years_extr = len(years_extr_all)
print("EXTR run years (for climatology):", years_extr_all)

# ---- 3.2.1 全部年份的月气候态 (clim_all) + 标准差 ----

clim_all_month = var.groupby("time.month").mean("time")  # (month,)
clim_all_std = var.groupby("time.month").std("time")

# 对齐 1–12 月（以防 month 坐标不完整）
clim_all_month = clim_all_month.reindex(month=months)
clim_all_std = clim_all_std.reindex(month=months)

# ---- 3.2.2 按 3–4 月日值的“春季最小值”选出最低 25% 的年份 ----

# 只取 3、4 月的日值
spring = var.sel(time=var.time.dt.month.isin([3, 4]))  # (time,)

# 每年 3–4 月的最小值
spring_min_by_year = spring.groupby("time.year").min("time")  # (year,)

spring_years = spring_min_by_year["year"].values
spring_min_values = spring_min_by_year.values

# 按最小值从小到大排序，取最底 25%
n_low = int(0.25 * len(spring_years))   # 200 年 → 50 年
n_low = max(n_low, 1)                   # 至少保证有 1 年

idx_sorted = np.argsort(spring_min_values)          # 升序
idx_low = idx_sorted[:n_low]                        # 最低的 n_low 年
low_ozone_years = spring_years[idx_low]             # 低臭氧年份列表

print(f"Selected {n_low} low-ozone years (bottom 25%):", low_ozone_years)

# ---- 3.2.3 低臭氧年份的月合成 climatology + 标准差 ----

var_low = var.sel(time=var.time.dt.year.isin(low_ozone_years))
clim_low_month = var_low.groupby("time.month").mean("time")
clim_low_std = var_low.groupby("time.month").std("time")

clim_low_month = clim_low_month.reindex(month=months)
clim_low_std = clim_low_std.reindex(month=months)

# 命名建议：
#   - "All-year climatology"（黑色虚线）
#   - "Low-ozone years composite (bottom 25%)"（黑色实线）
# 中文可以叫：
#   - “全样本气候态”
#   - “低臭氧年份合成（最低 25%）”


In [ ]:
# ========================================================
# Block 4 — 绘图：
#   - 4 年（0008/0013/0014/0019）的逐月部分柱 (彩色线)
#   - 全样本 200 年 climatology (黑虚线 + 浅灰包络)
#   - 低臭氧年份合成 climatology (黑实线 + 斜线灰色包络)
# ========================================================

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib import rcParams

# -------- 4.1 设置绘图风格（偏 Nature Geoscience） --------

rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "axes.linewidth": 0.8,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

fig, ax = plt.subplots(
    1, 1,
    figsize=(6.0, 3.8),  # 比较扁一点的版式，适合作为单 panel
    constrained_layout=True
)

months = np.arange(1, 13, dtype=int)
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# -------- 4.2 画 4 条特定年份的彩色线 --------

# 使用 tab10 调色板，保证可区分
colors_years = plt.cm.tab10(np.linspace(0, 1, len(years_special)))

for i, y in enumerate(years_special):
    ax.plot(
        months,
        monthly_pc_merged.sel(year=y).values,
        linestyle="-",
        linewidth=1.5,
        color=colors_years[i],
        label=f"Year {y:04d}"
    )

# -------- 4.3 画“全样本气候态” (黑虚线 + 浅灰包络) --------

# 浅灰填充 ±1σ
ax.fill_between(
    months,
    clim_all_month.values - clim_all_std.values,
    clim_all_month.values + clim_all_std.values,
    color="0.85",
    alpha=0.7,
    linewidth=0,
    zorder=0
)

# 黑色虚线气候态线
ax.plot(
    months,
    clim_all_month.values,
    linestyle="--",
    linewidth=1.8,
    color="black",
    label="All-year climatology"
)

# -------- 4.4 画“低臭氧年份合成气候态” (黑实线 + 斜线灰包络) --------

# 带斜线的灰色包络（facecolor=none + hatch）
ax.fill_between(
    months,
    clim_low_month.values - clim_low_std.values,
    clim_low_month.values + clim_low_std.values,
    facecolor="none",
    edgecolor="0.5",
    hatch="///",
    linewidth=0.0,
    zorder=1
)

ax.plot(
    months,
    clim_low_month.values,
    linestyle="-",
    linewidth=2.0,
    color="black",
    label="Low-ozone years composite (bottom 25%)"
)

# -------- 4.5 轴设置 --------

ax.set_xlim(1, 12)
ax.set_xticks(months)
ax.set_xticklabels(month_labels)

ax.set_ylabel("Partial O$_3$ column, 30–70 hPa (DU)")

# 让 y 轴刻度标签稍微大一点
for label in ax.get_yticklabels():
    label.set_fontsize(9)

# -------- 4.6 图例：手动构造更清晰的 legend --------

# 先为包络构造图例元素
patch_all = Patch(
    facecolor="0.85",
    edgecolor="none",
    label="All-year ±1σ"
)

patch_low = Patch(
    facecolor="none",
    edgecolor="0.5",
    hatch="///",
    label="Low-ozone ±1σ"
)

# 年份线（刚才已经用真实曲线加过 label 了）
handles_years = [
    Line2D([0], [0], color=colors_years[i], lw=1.5, label=f"Year {y:04d}")
    for i, y in enumerate(years_special)
]

# 气候态两条线（实线 & 虚线）
line_all = Line2D([0], [0], color="black", lw=1.8, ls="--", label="All-year climatology")
line_low = Line2D([0], [0], color="black", lw=2.0, ls="-", label="Low-ozone composite")

handles = handles_years + [line_all, patch_all, line_low, patch_low]

ax.legend(
    handles=handles,
    loc="best",
    frameon=False,
    fontsize=8,
    ncol=2
)

# -------- 4.7 标题、网格、保存 --------

ax.set_title("Polar cap (60–90°N) partial ozone column (30–70 hPa)")

ax.grid(False)  # Nature 系列一般不用显式网格，干净一点

out_dir = "/home/weiji/test_code/plots"
os.makedirs(out_dir, exist_ok=True)

fig_path_png = os.path.join(out_dir, "O3_column_monthly_4years_vs_climatology.png")
fig_path_pdf = os.path.join(out_dir, "O3_column_monthly_4years_vs_climatology.pdf")

plt.savefig(fig_path_png, dpi=300)
plt.savefig(fig_path_pdf)

plt.show()

print("Figure saved to:")
print("  ", fig_path_png)
print("  ", fig_path_pdf)
